In [ ]:
!nvidia-smi

Thu Mar  5 20:24:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# python3.12

In [ ]:
pip install wheel setuptools

In [ ]:
pip install torch==2.10.0 torchvision==0.25.0

Looking in indexes: https://download.pytorch.org/whl/cu130


In [ ]:
pip install timm 'accelerate>=0.26.0' qwen-vl-utils==0.0.14 transformers==5.0.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 55.6 MB/s eta 0:00:00


In [ ]:
pip install ms-swift==3.9.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.8/298.8 kB 10.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 12.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.6/449.6 kB 26.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 876.5/876.5 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 133.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 122.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!unzip -qq qwen_dataset.zip

In [ ]:
import pandas as pd

df = pd.read_json('qwen_dataset_train/dataset.jsonl', lines=True)
df.images = df.images.apply(lambda x: [img.replace('/Users/yuratomakov/innopolis/pdf2md/vie', '.') for img in x])
df.to_json('qwen_dataset/dataset_rel.jsonl', 'records', lines=True, index=False)

pd.read_json('qwen_dataset/dataset_rel.jsonl', lines=True).iloc[0].to_dict()
df.sample().iloc[0].to_dict()


{'messages': [{'role': 'system',
   'content': 'Ты точно извлекаешь пары ключ-значение из технических документов. Возвращай только валидный JSON без пояснений.'},
  {'role': 'user',
   'content': 'Тебе будет дано изображение страницы  PDF из опросного листа (технического документа на оборудование).\nИзвлеки спсиок характеристик и требований в JSON формате. Без комментариев, без пояснений, без Markdown.\n\nJSON format:\n[\n  {\n    "name": "<название параметра, которое запрашивает Заказчик>",\n    "request_value": "<значение парметра, которое запрашивает Заказчик>"\n  },\n  ...\n]\n\nСТРОГИЕ ПРАВИЛА:\n\n1) Верни ТОЛЬКО JSON. Любой текст вне JSON запрещён.\n\n2) name:\n   - Используй формулировку характеристики или требования из документа.\n   - Нормализуй переносы строк и пробелы, но НЕ меняй смысл.\n   - Если требование оформлено предложением (например: "… должен …"), используй ВЕСЬ текст требования как name, в одну строку.\n\n3) request_value:\n   - Для параметров, заданных в таблицах

In [ ]:
!CUDA_VISIBLE_DEVICES=0 \
swift sft \
    --model Qwen/Qwen3-VL-8B-Instruct \
    --freeze_vit True \
    --freeze_aligner True \
    --train_type lora \
    --dataset qwen_dataset_train/dataset.jsonl qwen_dataset_synt/dataset.jsonl \
    --val_dataset qwen_dataset_val/dataset.jsonl \
    --torch_dtype bfloat16 \
    --num_train_epochs 5 \
    --per_device_train_batch_size 1 \
    --per_device_eval_batch_size 1 \
    --learning_rate 1e-5 \
    --lora_rank 8 \
    --lora_alpha 32 \
    --target_modules all-linear \
    --gradient_accumulation_steps 16 \
    --eval_steps 10 \
    --save_steps 10 \
    --save_total_limit 5 \
    --logging_steps 10 \
    --max_length 6000 \
    --output_dir output_sft \
    --warmup_ratio 0.05 \
    --dataloader_num_workers 4 \
    --lr_scheduler_type constant \
    --use_hf=1


In [ ]:
model_chkp = Path('/content/output_sft/v0-20260305-203024/checkpoint-50')


In [ ]:
!zip -r qwen3-vl-4b-chkp.zip {model_chkp.absolute().as_posix()}


  adding: content/output_sft/v0-20260305-203024/checkpoint-50/ (stored 0%)
  adding: content/output_sft/v0-20260305-203024/checkpoint-50/additional_config.json (deflated 27%)
  adding: content/output_sft/v0-20260305-203024/checkpoint-50/scheduler.pt (deflated 62%)
  adding: content/output_sft/v0-20260305-203024/checkpoint-50/optimizer.pt (deflated 8%)
  adding: content/output_sft/v0-20260305-203024/checkpoint-50/README.md (deflated 65%)
  adding: content/output_sft/v0-20260305-203024/checkpoint-50/adapter_config.json (deflated 51%)
  adding: content/output_sft/v0-20260305-203024/checkpoint-50/rng_state.pth (deflated 26%)
  adding: content/output_sft/v0-20260305-203024/checkpoint-50/args.json (deflated 69%)
  adding: content/output_sft/v0-20260305-203024/checkpoint-50/adapter_model.safetensors (deflated 8%)
  adding: content/output_sft/v0-20260305-203024/checkpoint-50/trainer_state.json (deflated 71%)
  adding: content/output_sft/v0-20260305-203024/checkpoint-50/training_args.bin (defla